# 01. Data audit and cleaning

Input: the 9 raw Olist CSVs in `data/raw/`. Output: cleaned tables in `data/processed/` as parquet, ready for the SQL layer and every later phase.

The approach: load each table, audit it for missing values, duplicates, and values that cannot be true, then fix what is actually wrong and document why. Cleaning here only repairs table-level defects. Filters that depend on the question, such as keeping only delivered orders, belong to the analysis phases, so nothing a later question might need gets thrown away.

In [1]:
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")

orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW / "olist_order_reviews_dataset.csv")
customers = pd.read_csv(RAW / "olist_customers_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(RAW / "olist_geolocation_dataset.csv")
translation = pd.read_csv(RAW / "product_category_name_translation.csv")

tables = {
    "orders": orders, "order_items": order_items, "payments": payments,
    "reviews": reviews, "customers": customers, "products": products,
    "sellers": sellers, "geolocation": geolocation, "translation": translation,
}
rows_before = {name: len(df) for name, df in tables.items()}

## Audit

Three generic checks run on every table: shape, exact duplicate rows, and null counts per column. The targeted checks live in each table's own section below, because the interesting problems are specific: impossible date sequences in orders, several reviews for one order, untranslated categories in products.

In [2]:
for name, df in tables.items():
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    null_desc = ", ".join(f"{col}={n:,}" for col, n in nulls.items()) or "none"
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols, "
          f"duplicate rows={df.duplicated().sum():,}, nulls: {null_desc}")

orders: 99,441 rows x 8 cols, duplicate rows=0, nulls: order_approved_at=160, order_delivered_carrier_date=1,783, order_delivered_customer_date=2,965
order_items: 112,650 rows x 7 cols, duplicate rows=0, nulls: none
payments: 103,886 rows x 5 cols, duplicate rows=0, nulls: none
reviews: 99,224 rows x 7 cols, duplicate rows=0, nulls: review_comment_title=87,656, review_comment_message=58,247
customers: 99,441 rows x 5 cols, duplicate rows=0, nulls: none
products: 32,951 rows x 9 cols, duplicate rows=0, nulls: product_category_name=610, product_name_lenght=610, product_description_lenght=610, product_photos_qty=610, product_weight_g=2, product_length_cm=2, product_height_cm=2, product_width_cm=2
sellers: 3,095 rows x 4 cols, duplicate rows=0, nulls: none


geolocation: 1,000,163 rows x 5 cols, duplicate rows=261,831, nulls: none
translation: 71 rows x 2 cols, duplicate rows=0, nulls: none


The generic pass already shows the shape of the work. Geolocation carries 261,831 exact duplicate rows. Products has a block of 610 rows missing the category and the name and description lengths together, which points at one broken upstream feed rather than random gaps. The review comment nulls are customers who scored without writing text, so they are information, not damage. Orders has nulls in its progression dates, checked next against order status.

## Orders

Grain: one row per order.

**Issue 1: all five date columns load as text.** Choice: convert to datetime once here so every later phase gets real timestamps. No alternative worth debating.

**Issue 2: progression dates are null when the order never reached that step.** 2,965 orders lack a customer delivery date, but the nulls track order status: almost all belong to canceled, shipped, or unavailable orders. Only 8 delivered orders miss their delivery date. Options: drop those rows, impute a date, or keep the nulls. Choice: keep the nulls. Delivery analyses filter on status plus a present delivery date, so the 8 defective rows fall out exactly where they should, and imputing would fabricate the metric BQ3 is built on.

**Issue 3: 23 delivered orders record customer delivery before carrier hand-off.** One of the two timestamps is wrong and there is no way to tell which. Choice: keep them and note it. Delivery-time metrics compare delivery against purchase and estimated dates, and neither is involved in the contradiction. The 6 canceled orders that carry a delivery date stay too: status, not dates, decides what enters an analysis.

In [3]:
order_date_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders[order_date_cols] = orders[order_date_cols].apply(pd.to_datetime)

status_by_missing_delivery = (
    orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()
)
print(status_by_missing_delivery.to_dict())
print("purchase span:", orders["order_purchase_timestamp"].min().date(),
      "to", orders["order_purchase_timestamp"].max().date())

{'shipped': 1107, 'canceled': 619, 'unavailable': 609, 'invoiced': 314, 'processing': 301, 'delivered': 8, 'created': 5, 'approved': 2}
purchase span: 2016-09-04 to 2018-10-17


The nulls follow status as expected, and the purchase span confirms the documented September 2016 to October 2018 window. Orders is clean apart from what stays documented above.

## Order items

Grain: one row per physical unit in an order. `order_item_id` counts 1 to n within each order, so a two-unit order contributes two rows with the same product and price.

The audit found no broken values: no zero or negative prices, no negative freight. Two oddities are fine on inspection. 383 rows have freight of exactly 0, which is free shipping, not damage. 4 rows have a `shipping_limit_date` in 2020, past the end of the dataset, but the column is a contractual seller deadline rather than a shipment event, so a wrong deadline touches no delivery metric. The only change is parsing that date column.

In [4]:
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

print("price min/max:", order_items["price"].min(), "/", order_items["price"].max())
print("freight min/max:", order_items["freight_value"].min(), "/", order_items["freight_value"].max())
print("rows with freight 0:", (order_items["freight_value"] == 0).sum())

price min/max: 0.85 / 6735.0
freight min/max: 0.0 / 409.68
rows with freight 0: 383


## Payments

Grain: one row per payment transaction. One order can split across several rows, for example a credit card charge plus one or more vouchers, so payment analyses must aggregate to order level before joining.

**Issue 1: 3 rows have payment_type `not_defined` and a value of 0.00.** Options: keep or drop. Choice: drop. They record no money and no method, and keeping them would put a `not_defined` slice in every payment-method chart for the sake of 3 rows out of 103,886.

**Issue 2: 2 credit card rows show 0 installments.** A charge cannot have fewer than 1 installment. Choice: set them to 1, the smallest correction that makes the value possible.

**Kept: 6 voucher rows with value 0.** A fully applied voucher can leave a 0-value leg. They change no sums, so they stay.

In [5]:
payments = payments[payments["payment_type"] != "not_defined"].copy()
payments.loc[payments["payment_installments"] == 0, "payment_installments"] = 1

assert (payments["payment_installments"] >= 1).all()
print("rows:", len(payments), "| types:", payments["payment_type"].value_counts().to_dict())

rows: 103883 | types: {'credit_card': 76795, 'boleto': 19784, 'voucher': 5775, 'debit_card': 1529}


## Reviews

Raw grain: one row per review, which is not one row per order. 551 orders carry more than one review, typically a survey answered twice. On top of that, 814 rows reuse a `review_id` that also appears on a different order.

**Issue: BQ3 and BQ4 need exactly one score per order.** Options: aggregate at query time in every analysis, keep the first review, or keep the last. Choice: keep the review with the latest `review_answer_timestamp`, the customer's final word, and drop the rest here so every later join is one-to-one. That removes 551 rows. `review_id` stays non-unique where one id spans two orders, which is harmless because `order_id` is the join key everywhere; the SQL layer will key this table on `order_id`.

The comment columns stay even though the review model must not use text: 88% of customers wrote no comment title, and the presence of a comment is itself worth a look in EDA.

In [6]:
review_date_cols = ["review_creation_date", "review_answer_timestamp"]
reviews[review_date_cols] = reviews[review_date_cols].apply(pd.to_datetime)

reviews = (
    reviews.sort_values("review_answer_timestamp")
    .drop_duplicates("order_id", keep="last")
    .reset_index(drop=True)
)

assert reviews["order_id"].is_unique
print("rows:", len(reviews), "| score distribution:",
      reviews["review_score"].value_counts().sort_index().to_dict())

rows: 98673 | score distribution: {1: 11363, 2: 3131, 3: 8133, 4: 19038, 5: 57008}


## Products

Grain: one row per product.

**Issue 1: two column names are misspelled in the source, `product_name_lenght` and `product_description_lenght`.** Renamed to `_length` so nobody greps for a typo in six months.

**Issue 2: category names are Portuguese, and two categories are missing from the translation file.** `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos` exist in products but not in the translation table. Choice: add the two missing translations by hand, then merge the English name into products. Every later phase reads one English category column and never touches the translation file again, so the file itself is not saved as a processed table.

**Issue 3: 610 products have no category at all.** The same rows also miss the name and description lengths, one broken upstream feed. Options: drop them, label them, or keep the null. Choice: keep the null. These products still appear in real orders, so dropping them would silently delete sales; analyses that group by category will show them as an explicit unknown bucket.

**Kept: 2 products without physical measurements and 4 with weight 0.** No planned analysis uses weight or dimensions, and inventing measurements has no upside.

In [7]:
products = products.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length",
})

manual_translations = pd.DataFrame({
    "product_category_name": [
        "pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos",
    ],
    "product_category_name_english": [
        "pc_gamer", "portable_kitchen_food_preparers",
    ],
})
translation_full = pd.concat([translation, manual_translations], ignore_index=True)
products = products.merge(translation_full, on="product_category_name", how="left")

untranslated = products["product_category_name"].notna() & products["product_category_name_english"].isna()
assert untranslated.sum() == 0
print("rows:", len(products), "| categories:", products["product_category_name_english"].nunique(),
      "| products without category:", products["product_category_name_english"].isna().sum())

rows: 32951 | categories: 73 | products without category: 610


## Customers and sellers

Customers grain: one row per order-side customer record. `customer_id` changes on every order; `customer_unique_id` is the person. That distinction is the difference between a 0% and a 3% repeat rate, so it gets verified here rather than trusted.

Sellers grain: one row per seller.

Neither table has defects: no duplicate keys, no nulls. Zip code prefixes load as integers, which drops the leading zero of São Paulo prefixes on screen. They stay integers because every table stores them the same way, so joins are consistent, and the prefix is only ever used as a join key, never displayed.

In [8]:
persons = customers["customer_unique_id"].nunique()
repeaters = (customers["customer_unique_id"].value_counts() > 1).sum()
print(f"customer records: {len(customers):,} | unique persons: {persons:,} "
      f"| persons with >1 order: {repeaters:,} ({repeaters / persons:.1%})")
print(f"sellers: {len(sellers):,} across {sellers['seller_state'].nunique()} states")

customer records: 99,441 | unique persons: 96,096 | persons with >1 order: 2,997 (3.1%)
sellers: 3,095 across 23 states


## Geolocation

Raw grain: one row per geocoded address point, just over a million rows. 261,831 are exact duplicates and 42 points fall outside Brazil's bounding box, some in the middle of oceans.

The only thing any planned analysis needs from this table is one coordinate per zip prefix, for maps. Options: skip the table, keep it raw, or reduce it. Choice: drop the exact duplicates and the off-map points, then take the median latitude and longitude per zip prefix. Median rather than mean so a single bad geocode cannot drag a zip into the sea. The city and state columns are dropped: customers and sellers already carry their own, and the spellings here vary row to row.

278 customer zips and 7 seller zips have no geolocation row at all, so maps built on this table will miss those few points. Documented, not fixable from this data.

In [9]:
inside_brazil = (
    geolocation["geolocation_lat"].between(-33.8, 5.3)
    & geolocation["geolocation_lng"].between(-74.0, -34.7)
)
geolocation_zip = (
    geolocation[inside_brazil]
    .drop_duplicates()
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(geolocation_lat=("geolocation_lat", "median"),
         geolocation_lng=("geolocation_lng", "median"))
)

print("rows:", len(geolocation_zip), "| one row per zip prefix:",
      geolocation_zip["geolocation_zip_code_prefix"].is_unique)

rows: 19010 | one row per zip prefix: True


## Save cleaned tables

Everything downstream reads these parquet files, never the raw CSVs. The translation table is not saved because its content now lives inside `products`.

In [10]:
cleaned = {
    "orders": orders, "order_items": order_items, "payments": payments,
    "reviews": reviews, "customers": customers, "products": products,
    "sellers": sellers, "geolocation": geolocation_zip,
}
PROCESSED.mkdir(parents=True, exist_ok=True)
for name, df in cleaned.items():
    df.to_parquet(PROCESSED / f"{name}.parquet", index=False)

print("saved:", ", ".join(f"{name}.parquet" for name in cleaned))

saved: orders.parquet, order_items.parquet, payments.parquet, reviews.parquet, customers.parquet, products.parquet, sellers.parquet, geolocation.parquet


## Row counts before and after

Geolocation shrinks by design, one row per zip prefix instead of one per address point. Reviews loses the 551 extra reviews on multi-review orders. Payments loses the 3 empty `not_defined` rows. Every other table keeps every row: most issues were repaired in place or documented rather than deleted.

In [11]:
summary = pd.DataFrame({
    "rows_before": {name: rows_before[name] for name in cleaned},
    "rows_after": {name: len(df) for name, df in cleaned.items()},
})
summary.loc["geolocation", "rows_before"] = rows_before["geolocation"]
summary["rows_removed"] = summary["rows_before"] - summary["rows_after"]
print(summary.to_string())

             rows_before  rows_after  rows_removed
orders             99441       99441             0
order_items       112650      112650             0
payments          103886      103883             3
reviews            99224       98673           551
customers          99441       99441             0
products           32951       32951             0
sellers             3095        3095             0
geolocation      1000163       19010        981153
